# 👑 04. Supervisor 패턴 & 멀티에이전트 오케스트레이션 (5대 협업 프로토콜 & Blackboard Pattern)

단일 에이전트(Single Agent)에게 복잡한 시장 분석이나 데이터 수집 작업을 맡기면, 긴 도구 실행 로그가 누적되면서 **주의력 분산(Attention Drift)**과 **토큰 비용 폭증(Context Bloat)**이 발생합니다.

이를 해결하기 위해 현업의 프론티어 에이전트 시스템은 전체 기획을 총괄하는 **Supervisor(부모)**와 특정 전문 작업만 격리 수행하는 **Worker(자식)**로 분리된 **멀티에이전트 아키텍처**를 사용합니다.

부모와 자식 간의 **정보 불균형(Information Asymmetry)**을 원천 차단하는 **5대 협업 프로토콜**과 디스크 칠판 기반의 **Planning & Task Board 도구 엔진**을 직접 구현하고 실전 시나리오를 검증합니다.

```
👤 사용자: "온라인 쇼핑몰 노트북 시장 데이터를 분석해서 최적의 가성비 모델을 추천하고 종합 리포트를 작성해줘."
    ↓
👑 Supervisor (오케스트레이터 & Planning Tools)
    ├── 🛠️ Planning Tools (enter_plan, task_create, task_list, task_update, exit_plan)
    ├── 🔌 HierarchicalVisualizerMiddleware (app.middleware 기반 실시간 계층 로깅)
    ├── ⚡ Dynamic Context Pruning (Worker에게 필요한 Target File List & 지시문만 격리 주입)
    └── 👷 Worker (Market Analyst: target_file_list 자율 열람 + 가성비 점수 산출 + artifacts/ 보고서 생성)
         ├── 🛠️ Common Tools (app.tools.common: file_read, file_writer, grep_search)
         ├── 🔌 HierarchicalVisualizerMiddleware (자식 전용 들여쓰기 샌드박스 로깅)
         ├── 📄 artifacts/notebooks/generated/laptop_market_report.md (상세 분석 및 비교표 디스크 보존)
         └── 📤 [TASK REPORT] (부모에게는 5줄 요약 포인터만 반환)
```

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 내용 |
| :---:|:---|:---|
| **Step 0** | **환경 세팅** | 루트 경로 탐색, `.env` 로드, `app.tools`, `app.middleware` 임포트 |
| **Part 1** | **멀티에이전트 정보 불균형과 5대 협업 프로토콜** | 정보 불균형(Information Asymmetry) 4대 병목과 5대 협업 프로토콜 심층 분석 |
| **Part 2** | **Planning & Task Board 도구 엔진** | `enter_plan`, `task_create`, `task_list`, `task_update`를 통한 공유 칠판(Blackboard) 통제 |
| **Part 3** | **Sub-Agent 래핑 & 5대 프로토콜 구현체** | `invoke_sub_agent` Pydantic 정밀 명세 & `target_file_list` 다중 파일 슬라이싱 |
| **Part 4** | **Supervisor 조립 & [시나리오 1] 가성비 분석** | 노트북 시장 데이터(`laptops_market_data.json`) 분석 & 추천 리포트 발행 (Happy Path) |
| **Part 5** | **장애 복구 & [시나리오 2] Backtracking** | `[BLOCKER]` 발생 시 이전 단계로 되돌아가 계획을 수정하는 자가 치유(Self-Healing) |
| **Part 6** | **실무 요약 & AAWS 프로덕션 확장 로드맵** | 멀티에이전트 3대 황금률 및 `Scraper`/`Analyst` 파이프라인 확장 가이드 |


## 🛠️ Step 0. 환경 세팅

필요한 패키지를 로드하고 프로젝트 루트 경로 및 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.


In [ ]:
import os
import sys
import json
import time
import asyncio
import nest_asyncio
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# 1. 환경변수 로드
load_dotenv(override=True)

# 2. 프로젝트 루트 경로 자동 설정 (상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. LangChain 핵심 모듈 및 검증된 공용 도구(common.py, plan.py) 임포트
from pydantic import BaseModel, Field
from app.utils import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import MemorySaver
from app.utils import normalize_content

# app/tools 프로덕션 도구 및 app/middleware 계층형 시각화 미들웨어 임포트
from app.tools.common import file_read, file_writer, file_edit, grep_search
from app.tools.plan import enter_plan, exit_plan, task_create, task_list, task_update
from app.middleware import HierarchicalVisualizerMiddleware

print("✅ Core Modules, Production Tools (common + plan) & Visualizer Middleware Imported Successfully!")


---
## 📌 Part 1. 멀티에이전트 정보 불균형(Information Asymmetry)과 5대 협업 프로토콜

### 1.1 멀티에이전트 시스템의 최대 난제: 정보 불균형(Information Asymmetry)

LLM 에이전트를 실무 프로젝트에 투입할 때 발생하는 성능 저하와 실패의 90% 이상은 **"부모와 자식 간 정보의 비효율적 전달 및 컨텍스트 뒤섞임"**에서 기인합니다:

1. **Context Bloat & Token Explosion (컨텍스트 오염 및 비용 폭증)**:
   * 하위 에이전트가 크롤링한 원본 HTML, 수천 줄의 raw 데이터, 스크래치패드와 중간 오류 로그가 상위 오케스트레이터의 대화창에 무분별하게 누적되면 수만~수십만 토큰이 소모되고 API 비용이 기하급수적으로 증가합니다.
2. **Attention Drift & Hallucination (주의력 분산 및 환각 유발)**:
   * 컨텍스트 윈도우가 비대해지면 LLM의 주의 집중도가 급격히 분산되어 사용자의 최초 핵심 요구사항(가성비 산출 공식, 제약 조건 등)을 망각하거나 엉뚱한 분석에 갇혀버립니다.
3. **Black-box State Trap (블랙박스 상태 관리의 한계)**:
   * 런타임 진행 상태를 보이지 않는 메모리 딕셔너리(`state["messages"]`)로만 주고받으면, 에이전트가 어느 단계에서 왜 실패했는지 추적/디버깅이 불가능해집니다.
4. **Cascade Failure (단일 실패의 연쇄 붕괴 & Backtracking 부재)**:
   * 하위 작업에서 파일 손상이나 사이트 차단이 발생했을 때 상위 에이전트가 유연하게 계획을 재설계(Backtracking)하지 못하면 전체 시스템이 크래시되거나 무한 루프에 빠집니다.

---

### 1.2 정보 불균형 극복 아키텍처: Supervisor-Worker 구조 & Blackboard Pattern

이러한 문제를 원천 차단하기 위해 **역할 격리(Role Isolation)**와 **공유 칠판 패턴(Blackboard Pattern)**을 결합합니다:

* **Supervisor (부모 오케스트레이터)**: 사용자의 거시적 목표를 분석하여 계획을 수립하고, 하위 태스크를 디스패치하며, 최종 결과를 종합 보고합니다.
* **Worker (자식 전문 에이전트)**: 부모의 대화 기록에 일절 간섭받지 않는 격리된 샌드박스에서 도구를 자율 연쇄 실행하여 주어진 일감만 완수합니다.
* **Blackboard (공유 칠판)**: 에이전트 간 메모리를 직접 침범하지 않고, 파일 시스템 상의 구조화된 상태(`plan_state.json`, `task_state.json`)와 산출물 디렉토리(`artifacts/`)를 **단일 진실 공급원(SSOT)**으로 삼아 투명하게 상태를 공유합니다.

```
┌────────────────────────────────────────────────────────────────────────┐
│                        📋 Blackboard (공유 칠판)                        │
│     plan_state.json (거시 전략) / task_state.json (미시 체크리스트)      │
│     artifacts/ (영구 보존 산출물 저장소)                                │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ (상태 동기화 & 산출물 보존)
┌───────────────────────────────────▼────────────────────────────────────┐
│                    👑 Supervisor (Parent Orchestrator)                 │
│  - 사용자 요구사항 분석 & 칠판에 실행 계획/태스크 수립                 │
│  - ⚡ Dynamic Context Pruning (Worker에게 필요한 타겟 파일/일감만 슬라이싱)│
│  - 🔀 Backtracking (Worker의 [BLOCKER] 수신 시 계획 동적 재수립)      │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ invoke_sub_agent(target_file_list, instruction)
                                    ▼ (No Parent History Pollution)
┌────────────────────────────────────────────────────────────────────────┐
│                     👷 Worker Layer (Isolated Sandbox)                 │
│  - 🛡️ Prompt Layering (No-Chat 침묵 실행 & Fact-Only 보고 규약)        │
│  - 👶 Self-directed Gathering (인계받은 파일을 도구로 자율 탐색)       │
│  - 📄 Artifacts 디스크 생성 & [TASK REPORT] 요약 포인터만 반환          │
└────────────────────────────────────────────────────────────────────────┘
```

---

### 1.3 정보 불균형을 원천 차단하는 5대 협업 프로토콜 (Deep Dive)

| # | 프로토콜 명칭 | 핵심 원리 (Mechanism) | 해결하는 정보 불균형 문제 | 실습 구현 도구 / 코드 |
| :-: | :--- | :--- | :--- | :--- |
| **1** | **⚡ Dynamic Context Pruning**<br>(동적 맥락 가지치기) | 부모의 전체 대화 기록을 넘기지 않고, **`target_file_list`와 명확한 `task_instruction`만 슬라이싱**하여 Worker에 주입 | 자식의 Attention Drift 차단 및 토큰 낭비 제거 | `invoke_sub_agent`의 슬라이싱 인자 전달 |
| **2** | **👶 Self-directed Gathering**<br>(자율 팩트 수집) | 부모가 시시콜콜 지시하지 않고 목표만 부여 ➔ Worker가 내부 ReAct 루프에서 **도구(`file_read` 등)를 자율 연쇄 가동**하여 팩트 수집 | 상위 에이전트의 통제 오버헤드 감소 및 작업 자율성 | Worker의 `file_read`, `grep_search` 연쇄 |
| **3** | **🛡️ Prompt Layering**<br>(계층형 프롬프트 & 침묵 실행) | 부모와 자식의 Persona 분리. Worker에게 **"No-Chat(잡담/인사 금지), Fact-Only 실행 및 규격화된 포맷 출력"**을 강제 | 불필요한 대화 토큰 낭비 제거 및 응답 파싱 안정성 | Worker 전용 `worker_system_prompt` |
| **4** | **📄 Artifacts & Summary Pointer**<br>(산출물 격리 & 요약 보고) | 상세 분석 표나 대용량 데이터는 **`artifacts/` 디스크 파일로 저장**하고, 부모에게는 **5줄 `[TASK REPORT]` 요약 포인터만 반환** | 부모 컨텍스트 윈도우 오염 방지 (태스크 수십 번 반복 가능) | `file_writer` 저장 + `[TASK REPORT]` 반환 |
| **5** | **🔀 Backtracking & Self-Healing**<br>(백트래킹 자가 치유) | Worker 장애 발생 시 중단 없이 **`[BLOCKER: 원인]` 규약 반환** ➔ Supervisor가 **이전 계획 단계로 백트랙**하여 대안 데이터로 재시도 | 단일 실패의 연쇄 붕괴 방지 및 파이프라인 자가 치유 | `[BLOCKER]` 반환 ➔ 백업 카탈로그 재계획 루프 |


---
## 📌 Part 2. Planning & Task Board 도구 엔진 (Blackboard Pattern)

문자열 치환(`file_edit`) 방식은 LLM의 기억 오류로 인한 텍스트 매칭 실패 위험이 있습니다.
`frontier-agent-lab/app/tools/plan.py` 패턴을 계승하여, Supervisor 전용의 **구조화된 5대 Planning 도구**로 디스크 칠판(`plan_state.json`, `task_state.json`)을 조작합니다:

1. `enter_plan`: 거시적 실행 계획 등록 (`plan_state.json` 동기화)
2. `task_create`: 작업 단위 생성 (`task_state.json`에 Task ID 발급 및 `PENDING` 상태 등록)
3. `task_list`: 등록된 전체 태스크 현황 목록 조회
4. `task_update`: 태스크 상태 갱신 (`PENDING` ➔ `IN_PROGRESS` ➔ `COMPLETED` / `BLOCKED`)
5. `exit_plan`: 전체 계획 완료 선언 (`plan_state.json` 완료 처리)


In [ ]:
print("🛠️  app.tools.plan 5대 도구 (enter_plan, task_create, task_list, task_update, exit_plan) 준비 완료!")

### 📁 실습 데이터 및 산출물 보관 경로 (`artifacts/notebooks/`)
실습과 관련된 모든 원본 데이터셋은 `artifacts/notebooks/data/`에, 에이전트 생성 산출물은 **`artifacts/notebooks/generated/`**에 보관됩니다:
- `artifacts/notebooks/data/laptops_market_data.json`: 노트북 5종 스펙/가격 데이터 (시나리오 1용)
- `artifacts/notebooks/data/specs_criteria.json`: 가성비 점수 산출 기준 가이드 파일 (다중 파일 인계 실습용)
- `artifacts/notebooks/data/corrupted_catalog.bin`: 손상된 바이너리 파일 (시나리오 2 실패 유발용)
- `artifacts/notebooks/data/backup_laptops_catalog.json`: 백업 카탈로그 (시나리오 2 백트래킹용)
- `artifacts/notebooks/generated/laptop_market_report.md`: Worker가 생성하는 최종 마켓 리포트

In [ ]:
# 3. 실습용 노트북 데이터 및 아티팩트 디렉토리 초기화 (artifacts/notebooks/)
nb_workspace_dir = os.path.join(project_root, "artifacts", "notebooks")
data_dir = os.path.join(nb_workspace_dir, "data")
generated_dir = os.path.join(nb_workspace_dir, "generated")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(generated_dir, exist_ok=True)

# 1) 메인 노트북 시장 데이터 생성
laptops_data = [
    {"id": "LP-001", "brand": "레노버", "model": "아이디어패드 Slim 3 15IAH8", "price_krw": 599000, "cpu": "Intel Core i5-12450H", "ram_gb": 16, "ssd_gb": 512, "weight_kg": 1.62, "rating": 4.7, "review_count": 420, "category": "가성비/사무용"},
    {"id": "LP-002", "brand": "LG전자", "model": "2026 그램 프로 16", "price_krw": 1890000, "cpu": "Intel Core Ultra 7 155H", "ram_gb": 32, "ssd_gb": 1024, "weight_kg": 1.19, "rating": 4.9, "review_count": 890, "category": "초경량/프리미엄"},
    {"id": "LP-003", "brand": "ASUS", "model": "ROG 제피러스 G14", "price_krw": 2150000, "cpu": "AMD Ryzen 9 8945HS", "gpu": "RTX 4060", "ram_gb": 32, "ssd_gb": 1024, "weight_kg": 1.50, "rating": 4.8, "review_count": 310, "category": "게이밍/크리에이터"},
    {"id": "LP-004", "brand": "한성컴퓨터", "model": "올데이롱 TFX5470UC", "price_krw": 689000, "cpu": "AMD Ryzen 5 7530U", "ram_gb": 16, "ssd_gb": 512, "weight_kg": 1.40, "rating": 4.5, "review_count": 180, "category": "가성비/배터리특화"},
    {"id": "LP-005", "brand": "Apple", "model": "MacBook Air 13 M3", "price_krw": 1590000, "cpu": "Apple M3", "ram_gb": 16, "ssd_gb": 512, "weight_kg": 1.24, "rating": 4.9, "review_count": 1250, "category": "고효율/디자인"}
]
with open(os.path.join(data_dir, "laptops_market_data.json"), "w", encoding="utf-8") as f:
    json.dump(laptops_data, f, ensure_ascii=False, indent=2)

# 2) 다중 파일 인계 실습용 평가 기준 가이드 파일 생성
criteria_data = {
    "scoring_formula": "Value_Score = (RAM_GB * 10 + SSD_GB / 10 + Rating * 20) / (Price_KRW / 100000)",
    "recommendation_threshold": "Top 2 highest Value_Score models",
    "required_columns": ["ID", "Brand", "Model", "Price", "Score", "Category"]
}
with open(os.path.join(data_dir, "specs_criteria.json"), "w", encoding="utf-8") as f:
    json.dump(criteria_data, f, ensure_ascii=False, indent=2)

# 3) 예외 유발용 손상 바이너리 및 백업 카탈로그 생성
with open(os.path.join(data_dir, "corrupted_catalog.bin"), "w", encoding="utf-8") as f:
    f.write("[CORRUPTED_LEGACY_BINARY_BLOB_ERROR_0xDEADBEEF_CANNOT_PARSE]")

with open(os.path.join(data_dir, "backup_laptops_catalog.json"), "w", encoding="utf-8") as f:
    json.dump(laptops_data[:3], f, ensure_ascii=False, indent=2)

print(f"📁 Notebook Dataset Ready at: {nb_workspace_dir}")


---
## 📌 Part 3. Sub-Agent 래핑 & 정밀 Tool 명세 (5대 프로토콜 구현체)

### 3.1 LangChain Tool 작성 규약에 따른 `invoke_sub_agent` 설계
1. **`args_schema` (Pydantic BaseModel)**: 인자별 명확한 타입(`List[str]`, `str`)과 상세한 `Field(description=...)` 기술
2. **다중 파일 인계 (`target_file_list: List[str]`) — [Dynamic Context Pruning]**:
   * Supervisor는 Worker에게 필요한 메인 데이터 파일뿐만 아니라 기준표/설정 파일 목록을 함께 인계하여 컨텍스트를 격리합니다.
3. **자율 팩트 수집 — [Self-directed Gathering]**:
   * Worker는 인계받은 파일 목록을 순회하며 `file_read` 도구로 스스로 열람하고 분석합니다.
4. **Worker 전용 시스템 지침 — [Prompt Layering]**:
   * Worker는 대화형 인사 없이 팩트 분석을 수행하며, 실패 시 `[BLOCKER]`를 반환합니다.
5. **`Artifacts` 생성 & `Summary Pointer` 반환 — [Artifacts & Summary Pointer]**:
   * Worker 내부에서 `file_writer`로 `artifacts/notebooks/generated/laptop_market_report.md`에 상세 표를 작성하고, 반환값은 5줄 요약 `[TASK REPORT]`만 반환합니다.


In [ ]:
# 4. invoke_sub_agent Pydantic 스키마 정의 및 Worker 에이전트 구축

class InvokeSubAgentInput(BaseModel):
    task_instruction: str = Field(
        description="A clear, actionable step-by-step instruction detailing what the worker agent should analyze, calculate, and report."
    )
    target_file_list: List[str] = Field(
        description="List of relative file paths that the sub-agent needs to inspect (e.g. ['artifacts/notebooks/data/laptops_market_data.json', 'artifacts/notebooks/data/specs_criteria.json'])."
    )
    subagent_role: str = Field(
        default="Market Analyst",
        description="The persona and specialty of the sub-agent (e.g. 'Market Analyst', 'Data Specialist')."
    )

def run_worker_agent(task_instruction: str, target_file_list: List[str], role: str = "Market Analyst") -> str:
    """Sub-Agent를 독립 컨텍스트에서 실행하고 최종 보고서를 반환합니다."""
    worker_llm = init_chat_model(model="gemini-3.7-flash", temperature=0.0)
    
    file_list_str = ", ".join(target_file_list)
    
    # Prompt Layering: Worker 전용 Fact-Only / No-Chat 지침
    worker_system_prompt = f"""You are a specialized sub-agent ({role}).
You operate inside an isolated sandbox. You MUST follow these strict protocols:
1. No chit-chat or conversational greeting.
2. Read the assigned target files ({file_list_str}) using `file_read`.
3. If any critical data file is corrupted, binary, or not a valid JSON catalog, immediately return:
   [BLOCKER: Corrupted or non-JSON file format - Backup catalog required]
4. If valid, perform a thorough product analysis (compare price vs specs vs rating, compute value score)
   and write a detailed markdown report into 'artifacts/notebooks/generated/laptop_market_report.md' using `file_writer`.
   Include a Markdown comparison table and top 2 recommended models.
5. Finally, return ONLY a concise 5-line summary formatted EXACTLY as:
   [TASK REPORT]
   - Status: SUCCESS | FAILED | BLOCKER
   - Target Files: {file_list_str}
   - Artifacts Created: artifacts/notebooks/generated/laptop_market_report.md
   - Summary: (1-2 sentence core finding: top value model and price range)
   - Issues: None (or description of blockers)
"""

    # 자식 전용 미들웨어 주입 (is_subagent=True)
    worker_middleware = [HierarchicalVisualizerMiddleware(agent_role=role, is_subagent=True)]

    worker = create_agent(
        model=worker_llm,
        tools=[file_read, file_writer, grep_search], # 검증된 common 도구 장착!
        system_prompt=worker_system_prompt,
        middleware=worker_middleware
    )
    
    # Dynamic Context Pruning: 부모 대화 이력 없이 순수 지시문만 주입
    prompt = f"Target File List: {target_file_list}\nInstruction: {task_instruction}"
    result = worker.invoke({"messages": [HumanMessage(content=prompt)]})
    return result["messages"][-1].content


@tool(args_schema=InvokeSubAgentInput)
def invoke_sub_agent(task_instruction: str, target_file_list: List[str], subagent_role: str = "Market Analyst") -> str:
    """Forks an isolated, sandboxed sub-agent with a dedicated worker persona to execute focused discovery or data analysis tasks on target files.

    Use this tool when you need to delegate heavy file reading, multi-file cross-referencing, calculation, or report generation to a specialist worker.

    The sub-agent operates in an isolated ReAct loop, writes detailed artifacts directly to disk, and returns only a concise 5-line summary report.

    Args:
        task_instruction: Specific task directive for the sub-agent.
        target_file_list: List of file paths for the sub-agent to inspect.
        subagent_role: Role persona assigned to the sub-agent.

    Returns:
        A formatted `[TASK REPORT]` string containing Status (SUCCESS/BLOCKER), Target files, Artifacts Created, Summary, and Issues.
    """
    return run_worker_agent(task_instruction, target_file_list, subagent_role)

print("✅ `invoke_sub_agent` Tool with Pydantic Schema & target_file_list Ready!")


---
## 📌 Part 4. Supervisor 에이전트 조립 및 [시나리오 1: 정상 가성비 분석] 실행

Supervisor 에이전트는 사용자의 목표를 분석하여 다음과 같이 행동합니다:
1. `enter_plan` 및 `task_create`로 작업 보드(Task Board) 초기화
2. `invoke_sub_agent`를 호출하여 Worker에게 다중 파일 목록과 일감 디스패치
3. Worker의 `[TASK REPORT]` 수신 후, `task_update(task_id, 'COMPLETED')`로 원터치 갱신
4. `exit_plan`으로 계획 종료 선언 및 최종 비즈니스 요약 결과 보고


In [ ]:
# 5. Supervisor 에이전트 구축 및 시나리오 1 실행

supervisor_llm = init_chat_model(model="gemini-3.7-flash", temperature=0.0)

supervisor_system_prompt = """You are the Chief Supervisor Agent managing an automated e-commerce market research project.
When a user request arrives, execute the following protocol strictly in order:

1. [Macro Planning]:
   - Call `enter_plan` with plan name and steps.
   - Call `task_create` to register sub-tasks (e.g. 1. Inspect data & criteria, 2. Run analysis, 3. Generate summary).

2. [Task Delegation]:
   - Delegate the inspection of target files by calling `invoke_sub_agent`.
   - Pass the exact `target_file_list` (e.g. ['artifacts/notebooks/data/laptops_market_data.json', 'artifacts/notebooks/data/specs_criteria.json']).

3. [Task Board Update & Backtracking]:
   - Once `invoke_sub_agent` returns, check if it was SUCCESS or BLOCKER.
   - If SUCCESS: Call `task_update(task_id, 'COMPLETED')`.
   - If BLOCKER: Backtrack! Call `task_update(task_id, 'BLOCKED')`, create a fallback task, and call `invoke_sub_agent` on the alternative backup catalog.

4. [Conclusion]:
   - Call `exit_plan` to conclude.
   - Summarize the final outcome clearly for the user, highlighting the recommended laptop models and mentioning the created artifacts.
"""

# Supervisor 전용 미들웨어 및 도구 장착
supervisor_middleware = [HierarchicalVisualizerMiddleware(agent_role="Supervisor", is_subagent=False)]

supervisor_agent = create_agent(
    model=supervisor_llm,
    tools=[enter_plan, exit_plan, task_create, task_list, task_update, invoke_sub_agent, file_read],
    system_prompt=supervisor_system_prompt,
    middleware=supervisor_middleware
)

# -----------------------------------------------------------------------------
# [시나리오 1] 정상 노트북 시장 데이터 가성비 분석 요청 실행
# -----------------------------------------------------------------------------
print("🚀 [시나리오 1 시작] 노트북 시장 데이터 가성비 분석 요청...")

user_prompt_1 = """우리 회사의 'artifacts/notebooks/data/laptops_market_data.json' 시장 상품 데이터와 'artifacts/notebooks/data/specs_criteria.json' 기준표를 함께 분석해줘.
Task Board에 계획을 등록하고, Market Analyst에게 분석을 지시한 뒤, 
가격대별 추천 모델 및 비교표가 담긴 상세 보고서를 'artifacts/notebooks/generated/laptop_market_report.md'에 저장하고 최종 브리핑을 해줘."""

response_1 = supervisor_agent.invoke({
    "messages": [HumanMessage(content=user_prompt_1)]
})

print("="*80)
print("👑 [Supervisor 최종 답변]:")
print("="*80)
print(normalize_content(response_1["messages"][-1].content))


In [ ]:
# 5.1 시나리오 1 결과 검증 (Task Board 상태 & Artifacts 확인)

print("📋 [Task Board: task_state.json 내용 확인]")
print("-" * 50)
print(task_list.invoke({}))

print("\n" + "=" * 50)
print("📄 [Worker가 생성한 Artifact: artifacts/notebooks/generated/laptop_market_report.md]")
print("-" * 50)
report_content = file_read.invoke({"file_path": "artifacts/notebooks/generated/laptop_market_report.md"})
print(report_content)

---
## 📌 Part 5. 장애 대응 & [시나리오 2: Backtracking 자가 치유]

### 5.1 Backtracking (백트래킹 자가 치유) 매커니즘
Worker가 샌드박스 내부에서 데이터 손상이나 접근 불가를 마주했을 때:
1. 시스템 전체가 에러로 중단되지 않고 표준 예외 규약인 `[BLOCKER: 원인]` 텍스트를 반환합니다.
2. Supervisor는 이 보고를 수신한 뒤, **이전 계획 단계로 백트랙(Backtrack)**합니다.
3. Supervisor는 실패한 태스크 상태를 `BLOCKED`로 처리하고, 백업 카탈로그(`artifacts/notebooks/data/backup_laptops_catalog.json`)를 타겟으로 지정하여 2차 시도를 수행함으로써 파이프라인을 자가 치유(Self-Healing)합니다.


In [ ]:
# 6. [시나리오 2] 손상 파일 분석 실패 ➔ 백업 카탈로그로 Backtracking 자가 치유 실행

print("🚀 [시나리오 2 시작] 손상된 카탈로그 분석 실패 ➔ 백업 데이터로 백트래킹...")

user_prompt_2 = """신규 입고 데이터인 'artifacts/notebooks/data/corrupted_catalog.bin' 파일의 상품 데이터를 분석해줘.
만약 파일 손상이나 포맷 오류로 분석이 불가능하다면 멈추지 말고 백업 파일인 'artifacts/notebooks/data/backup_laptops_catalog.json'을 찾아 백트래킹 분석을 완수해줘."""

response_2 = supervisor_agent.invoke({
    "messages": [HumanMessage(content=user_prompt_2)]
})

print("="*80)
print("👑 [Supervisor 최종 답변]:")
print("="*80)
print(normalize_content(response_2["messages"][-1].content))

In [ ]:
print("\n" + "="*50)
print("📋 [자가 치유 완료된 Task Board 최종 상태]")
print("="*50)
print(task_list.invoke({}))

---
## 📌 Part 6. 실무 요약 & AAWS 프로덕션 확장 로드맵

### 💡 정보 불균형 극복을 위한 멀티에이전트 설계 3대 황금률
1. **메모리(Messages)로 소통하지 말고 구조화된 Task Board로 소통하라**:
   * `enter_plan`, `task_create`, `task_update`를 통해 세션 단절과 환각 없이 작업 진행도를 투명하게 통제합니다.
2. **세부 결과는 `artifacts/`에 영구 보존하고, 부모에게는 요약 포인터만 반환하라**:
   * 부모의 컨텍스트를 오염시키지 않아 수십 번의 태스크 반복에도 토큰 비용이 선형 증가하지 않습니다.
3. **Worker에게는 침묵 실행(No-Chat)과 Backtracking 규약을 강제하라**:
   * 실패 시 핑퐁 대화 대신 `[BLOCKER]`를 반환하여 부모가 전략을 동적으로 재설계할 수 있게 합니다.

---

### 🚀 AAWS 프로덕션 확장 안내 (`app/agents/supervisor.py`)

우리가 실습에서 구현한 5대 협업 프로토콜은 AAWS 프로덕션 환경에 다음과 같이 1:1로 확장 적용됩니다:

```
👑 Supervisor Agent (app/agents/supervisor.py)
 ├── 🔌 Logging / HITL Middleware
 ├── 🛠️ Planning Tools (enter_plan, task_create, task_update)
 ├── 🕷️ The Scraper (app/agents/scraper.py)
 │    └── 사이트 탐색 + 코드 작성/실행 ➔ artifacts/에 데이터 & extraction_plan.json 저장
 └── 📊 The Analyst (app/agents/analyst.py)
      └── 수집 데이터 통계 분석 + 시각화 차트 ➔ artifacts/에 종합 분석 보고서 저장
```
